# Assignment · Your own agent on the wildfire simulator

**CADS workshop follow-up · How to Build Agents for Research**

You will build the agent from the closing session, point it at the fire simulator, and ask it questions the workshop never asked. Every exercise has three parts, always in this order:

1. **Check first** — you compute the answer yourself with the simulator (no agent). Most of the code is written; you fill in the blanks marked `____`.
2. **Ask the agent** — you write the question in English and run `ask(...)`.
3. **Compare** — you write two or three sentences in the *Your answer* cell.

Seven exercises, each one habit from the workshop: **loop · trace · guard · debug · check · many runs**.

**How to fill a blank.** `____` is a placeholder. Replace the whole thing, including the underscores, with what the comment asks for. Then run the cell (Shift+Enter). If you see `NameError: name '____' is not defined`, a blank is still there.

**Before you start (once).** Install the Ollama app, run `ollama pull qwen2.5:7b-instruct` in a terminal, and put this notebook, `fire_sim.py` and `agent_tools.py` in one folder. Run the cells in order.

> **Prepared version:** All independent simulation blanks and agent question prompts have been completed. Independent results were calculated from the supplied `fire_sim.py` and `agent_tools.py` with default seed=0. **Agent results are not fabricated:** exercises 1–4 and 6 still require running the actual Ollama model, inspecting traces, and filling in agent-specific observations.

> **Google Colab setup:** Upload `fire_sim.py` and `agent_tools.py` using the Files sidebar before running the first setup cell. **Important:** the original notebook's `llm()` calls `localhost:11434`, which refers to the Colab server, not your personal computer. Installing Ollama on your laptop does not make it available to Colab automatically. To run agent cells in Colab, you must configure an accessible model endpoint (or run the notebook locally with Ollama). The independent check cells work in Colab without Ollama.


**GitHub portfolio note:** This notebook contains reproducible simulator checks, but does **not** claim that an AI agent was run. The `RUN_AGENT` switches default to `False` because the workshop uses an Ollama server at `localhost:11434`, which is not automatically available in hosted Colab. For a completed agent experiment, configure a reachable model, set each switch to `True`, inspect its real traces, and fill in the agent-comparison responses. The workshop's substring-based `guard()` is a teaching example, **not** a secure sandbox for executing untrusted generated code.

## Google Colab model setup (required for agent exercises)
Run the next cell **once** before the original setup. It installs and launches Ollama inside the Colab runtime and downloads the workshop model. This may take several minutes and multiple GB of storage. If it fails due to Colab resource limits, run the notebook locally instead. The agent executes generated Python; use only trusted workshop code and an isolated runtime.


In [ ]:
# Colab-only Ollama installation and startup (run before the original setup cells).
import os, sys, subprocess, time, urllib.request, shutil
if 'google.colab' in sys.modules:
    if shutil.which('ollama') is None:
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
    # Start only if not already listening.
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=3)
    except Exception:
        log = open('/tmp/ollama-colab.log', 'a')
        subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        for _ in range(30):
            try:
                urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
                break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError('Ollama server did not start. Check /tmp/ollama-colab.log')
    subprocess.run(['ollama', 'pull', 'qwen2.5:7b-instruct'], check=True)
    print('Ollama ready in this Colab runtime.')
else:
    print('Not running in Colab. Start Ollama locally and pull qwen2.5:7b-instruct.')


## Setup · run these three cells as they are

The first cell loads the simulator and draws one fire so you can see the map. The second loads the agent from the closing session (`step5_check.py`), unchanged except that the model's code can call the three simulator tools. The third is a helper, `ask()`, that runs the agent and prints its answer.

In [ ]:
# Colab dependency check: upload fire_sim.py and agent_tools.py using the Files sidebar.
import os
for filename in ("fire_sim.py", "agent_tools.py"):
    if not os.path.isfile(filename):
        raise FileNotFoundError(f"Upload {filename} into the Colab Files sidebar first.")
print("Supporting simulator files found.")

In [ ]:
from fire_sim import *            # the simulator: simulate(), structures_lost(), area_burned(), wilson(), show()
from agent_tools import *         # the three tools the agent may call: describe_landscape(), run_weather(), run_plan()
import numpy as np, json, re, subprocess, sys, time, urllib.request

one_fire = simulate(n=1)          # one fire, today's weather: wind 6 m/s toward E, dryness 0.9, 55 steps
show(one_fire)
print(TOOLS_DOC)

In [ ]:
# The agent. You do not need to change anything here. Read the comments; you wrote this loop in the workshop.
MODEL = "qwen2.5:7b-instruct"     # change to a name shown by `ollama list` if different

def llm(messages):                # the model: conversation in, text out (runs on your laptop)
    req = urllib.request.Request("http://localhost:11434/v1/chat/completions",
                                 json.dumps({"model": MODEL, "messages": messages}).encode(),
                                 {"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=600))["choices"][0]["message"]["content"]

def run_python(code):             # the tool: run the model's code with the simulator tools already imported
    open("_step.py", "w").write("from agent_tools import *\n" + code)
    p = subprocess.run([sys.executable, "_step.py"], capture_output=True, text=True, timeout=120)
    return (p.stdout + p.stderr).strip()[-2500:] or "(no output)"

RULES = ("You study a wildfire simulator by writing Python. These functions are already imported and are the ONLY way "
         "to get numbers:\n" + TOOLS_DOC + "\n"
         "Each reply holds exactly one ```python block that calls them and print()s the results; I will run it and send "
         "back what it printed. Never invent a number. Look at the landscape first. Compare settings on the same seed and "
         "the same n. When you have the answer, reply with no code block, starting with ANSWER:, and quote the numbers.")

FORBIDDEN = ["os.remove", "rmtree", "os.system", "subprocess", "urllib", "requests", "socket", "to_csv", ".write(", "pip install", "open("]
def guard(code):                  # guard 1: what it may not do
    hits = [w for w in FORBIDDEN if w in code]
    if hits:
        return f"BLOCKED: the code uses {hits}. Only call the simulator tools and print."
NUDGE = "Reply with exactly one ```python block to run, or with no code starting with ANSWER:."

def trace(label, text):           # the trace: a labelled box
    print(f"\n┌─ {label}\n" + "\n".join("│ " + l for l in str(text).strip().splitlines()) + "\n└" + "─" * 70)

def run_agent(task, max_steps=8, save="trace.json"):
    messages = [{"role": "system", "content": RULES}, {"role": "user", "content": task}]
    for step in range(max_steps):
        t0 = time.time()
        reply = llm(messages)                                                      # 1. decide
        messages.append({"role": "assistant", "content": reply})
        trace(f"step {step + 1} · MODEL ({time.time() - t0:.0f}s)", reply)         # door 1
        code = re.search(r"```(?:python|py)?\n(.*?)```", reply, re.S)              # 2. read
        if not code:
            if reply.strip().startswith("ANSWER:"):
                break                                                              # guard 3: this counts as done
            trace(f"step {step + 1} · LOOP", NUDGE); messages.append({"role": "user", "content": NUDGE}); continue
        out = guard(code.group(1)) or run_python(code.group(1))                    # 3. act, if the guard allows
        trace(f"step {step + 1} · TOOL OUTPUT", out)                                # door 2
        messages.append({"role": "user", "content": "OUTPUT:\n" + out})            # 4. feed back
    else:
        reply = None                                                               # guard 2: out of steps
    json.dump(messages, open(save, "w"), indent=1)
    return reply

print("agent ready, using", MODEL)

In [ ]:
def ask(question, save="trace.json"):
    """Run the agent on a question, show the trace, and print the final answer on its own line."""
    answer = run_agent(question, save=save)
    print("\n" + "=" * 78 + "\nAGENT SAYS:", answer, "\n" + "=" * 78)
    return answer

# Test that the model answers at all (takes a few seconds). You should see a short greeting.
try:
    print(llm([{"role": "user", "content": "Say hello in one sentence."}]))
except Exception as exc:
    print("AI model not connected:", type(exc).__name__)
    print("Independent simulation checks can run. Agent runs need a reachable Ollama endpoint.")

## Exercise 0 · The baseline, by hand (no agent yet)

Under today's weather, run 100 fires and find out how bad things are on average. Fill the blanks.

*Self-check:* mean structures lost should be between 30 and 33 (of 36); the share of runs reaching the town should be above 0.9.

In [ ]:
arrival = simulate(n=100)                       # run 100 fires at once
lost = structures_lost(arrival)                   # one number per fire: how many of the 36 buildings burned
print("mean structures lost:", round(lost.mean(), 1), "of 36")

reached = (lost > 0).sum()                        # how many of the fires reached the town at all
lo, hi = wilson(reached, 100)                    # 95% interval for that share; second argument = number of fires
print("runs reaching the town:", reached, "of 100 →", reached / 100, " interval", round(lo, 2), "to", round(hi, 2))

print("mean area burned:", round(area_burned(arrival).mean()), "hectares")

**Your answer (Ex 0 — independently verified):** Mean lost = **31.56 of 36**; share reaching town = **96/100 = 0.96** (95% Wilson interval **0.90 to 0.98**); area = **2537 ha** (rounded).

## Exercise 1 · Which wind direction is most dangerous for the town?

**Habit: the loop, and a check written first.**

The eight directions are in `COMPASS` (E, SE, S, SW, W, NW, N, NE; the direction the wind blows *toward*). `run_weather(toward=d)` runs 100 fires with that wind and returns a dictionary of summary numbers.

*Self-check:* one direction should be far worse than the rest, and several directions should give 0.0.

In [ ]:
# CHECK FIRST (no agent)
for d in COMPASS:
    result = run_weather(toward=d)             # the loop variable
    print(d, "→ mean structures lost:", result["mean_structures_lost"])

Now ask the agent. Write the question in plain English inside the quotes. A good question says **what to try** (all eight directions), **what to keep fixed** (everything else as today), **how to measure** (mean structures lost over 100 runs), and **how to finish** (`ANSWER: <direction>` plus the numbers).

In [ ]:
# AI-agent run requires a reachable model endpoint.
RUN_AGENT = False  # set True after configuring Ollama or a compatible endpoint
if RUN_AGENT:
    question_1 = """
    Compare all eight wind directions using run_weather(toward=d, n=100, seed=0). Keep today's speed and dryness fixed. Report the mean structures lost for each direction and finish with ANSWER: followed by the most dangerous direction.
    """
    answer_1 = ask(question_1, save="trace_ex1.json")
else:
    print("AI-agent trace pending; see the independently computed result above.")


**Your answer (Ex 1):** My independent check says **E** is worst (31.6 structures lost on average), followed by SE (21.9) and NE (8.8); S, SW, W, NW, N all yielded 0.0. **Agent comparison: pending actual run; inspect `trace_ex1.json` before completing.**

## Exercise 2 · Does a stronger wind make the fire worse?

**Habit: read the trace.** This question has a trap: *worse for whom?* The simulator reports two things, buildings lost and hectares burned. Compute both before you ask.

*Self-check:* as speed goes up, one of the two numbers goes up and the other goes down.

In [ ]:
# CHECK FIRST (no agent)
for speed in [2, 4, 6, 8, 10, 12]:
    result = run_weather(speed=speed)              # the loop variable
    print(speed, "m/s → lost:", result["mean_structures_lost"], "  area:", result["mean_area_burned_ha"], "ha")

Ask the agent the question **without telling it which number to use**. Say: wind speeds 2 to 12, today's direction and dryness, 100 runs each, same seed; finish with `ANSWER: yes, no or depends`, with the numbers used.

In [ ]:
# AI-agent run requires a reachable model endpoint.
RUN_AGENT = False  # set True after configuring Ollama or a compatible endpoint
if RUN_AGENT:
    question_2 = """
    Test wind speeds 2, 4, 6, 8, 10, and 12 m/s using 100 simulations each, seed=0, with today's wind direction and dryness. Is stronger wind worse? Finish with ANSWER: yes, no, or depends, supported by numbers.
    """
    answer_2 = ask(question_2, save="trace_ex2.json")
else:
    print("AI-agent trace pending; see the independently computed result above.")


**Your answer (Ex 2):** My check finds that mean building loss generally increases from **31.3** at 2 m/s to **33.5** at 12 m/s, while mean burned area falls from **2655 ha** to **1846 ha**. Thus “worse” **depends on the metric**. **Agent trace and whether it reported one or both metrics: pending actual run.**

## Exercise 3 · Which crossings should the crew close? The crew can close at most two.

**Habit: guard rails.** Open `agent_tools.py` and look at `run_plan`: if you ask for three crossings it refuses and says why. That rule lives *inside the tool*, so the model cannot talk its way past it.

There are seven allowed options: nothing; north; middle; south; north+middle; north+south; middle+south. Try them all.

*Self-check:* one two-crossing option should bring the mean loss below 3.

In [ ]:
# CHECK FIRST (no agent)
options = [[], ["north"], ["middle"], ["south"], ["north", "middle"], ["north", "south"], ["middle", "south"]]
for closed in options:
    result = run_plan(closed=closed)                # the loop variable
    print(closed, "→ mean structures lost:", result["mean_structures_lost"])

# Now try the forbidden one and read what comes back:
print(run_plan(closed=["north", "middle", "south"]))

Ask the agent which crossing(s) to close, at most two, comparing every allowed option with 100 runs on the same seed; finish with `ANSWER: <crossings>` and the numbers. **This trace is the one you hand in**, so keep `save="trace.json"`.

In [ ]:
# AI-agent run requires a reachable model endpoint.
RUN_AGENT = False  # set True after configuring Ollama or a compatible endpoint
if RUN_AGENT:
    question_3 = """
    Compare all seven permitted options for closing zero, one, or two crossings. Use run_plan with n=100 and seed=0 for every option, keeping today's weather fixed. The crew may close at most two crossings. Report the mean structures lost for each option and finish with ANSWER: the best crossing combination.
    """
    answer_3 = ask(question_3, save="trace.json")
else:
    print("AI-agent trace pending; see the independently computed result above.")


**Your answer (Ex 3):** My check finds **middle + south** best, with **1.9** structures lost on average, versus **31.6** with no closure. A three-crossing request is blocked by `run_plan` (maximum two). **Whether the agent tried three crossings and its response: pending actual trace.**

## Exercise 4 · Up to what dryness does closing the middle crossing still halve the loss?

**Habit: a check that may disagree with the agent.** In the workshop, a plan was approved only if it cut the mean loss to half of doing nothing. For each dryness from 0.6 to 1.3, compute the loss with the crossing open and with it closed, and the ratio closed ÷ open.

*Self-check:* the ratio should grow with dryness and cross 0.5 somewhere between 0.9 and 1.0.

In [ ]:
# CHECK FIRST (no agent)
for dryness in [0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]:
    open_loss   = run_weather(dryness=dryness)["mean_structures_lost"]
    closed_loss = run_plan(closed=["middle"], dryness=dryness)["mean_structures_lost"]     # which crossing?
    ratio = closed_loss / open_loss if open_loss > 0 else 0
    print(dryness, "→ open:", open_loss, " closed:", closed_loss, " ratio:", round(ratio, 2))

Ask the agent: up to what dryness does closing the middle crossing still cut the mean loss to at most half of doing nothing? Dryness 0.6 to 1.3 in steps of 0.1, 100 runs each, same seed; finish with `ANSWER: <dryness>` and the ratios.

In [ ]:
# AI-agent run requires a reachable model endpoint.
RUN_AGENT = False  # set True after configuring Ollama or a compatible endpoint
if RUN_AGENT:
    question_4 = """
    For dryness values 0.6 through 1.3 in increments of 0.1, compare run_weather against run_plan(closed=["middle"]) using n=100 and seed=0 for both. Compute the ratio of mean structures lost with the crossing closed versus open. Find the highest tested dryness at which the ratio is at most 0.5. Finish with ANSWER: the threshold and all ratios.
    """
    answer_4 = ask(question_4, save="trace_ex4.json")
else:
    print("AI-agent trace pending; see the independently computed result above.")


**Your answer (Ex 4):** My threshold is **0.9** (ratio **0.40**); at dryness **1.0** the ratio is **0.60**, so the reduction is no longer at least half. Ratios at dryness 0.6–1.3: **0.00, 0.11, 0.21, 0.40, 0.60, 0.81, 0.89, 0.97**. **Agent comparison: pending actual run.**

## Exercise 5 · Find the flaw in a trace (no code to write)

**Habit: debugging at the two doors.** Below is a trace from an agent that was asked Exercise 3's question. Its answer is confident and the trace looks clean. Read the code it wrote (the first `MODEL` box) slowly.

In [ ]:
BAD_TRACE = """
┌─ step 1 · MODEL (6s)
│ Comparing the options.
│ ```python
│ print('none', run_plan([], n=10)['mean_structures_lost'])
│ print('middle', run_plan(['middle'], n=10, seed=1)['mean_structures_lost'])
│ print('middle+south', run_plan(['middle','south'], n=10, seed=2)['mean_structures_lost'])
│ print('north+middle', run_plan(['north','middle'], n=10, seed=3)['mean_structures_lost'])
│ ```
└──────────────────────────────────────────────────────────────────────
┌─ step 1 · TOOL OUTPUT
│ none 32.4
│ middle 9.1
│ middle+south 3.6
│ north+middle 10.8
└──────────────────────────────────────────────────────────────────────
┌─ step 2 · MODEL (5s)
│ ANSWER: middle and south. Closing middle alone leaves 9.1 structures lost; adding south brings it to 3.6,
│ and north+middle (10.8) is worse than middle alone, so the north crossing should stay open.
└──────────────────────────────────────────────────────────────────────
"""
print(BAD_TRACE)

**Your answer (Ex 5):** (1) The flawed trace used only **10** fires per option rather than the **100** used in the baseline, giving a noisier comparison. (2) It used **different seeds** (0, 1, 2, 3), making comparisons less controlled, and omitted three permitted options. (3) The claimed **10.8 versus 9.1** does not establish that north+middle is worse than middle alone; with the same seed and 100 runs, my check gives **12.2** for north+middle versus **12.6** for middle alone. The key ignored rule was **check** (independent verification), and the comparison was not fair.

## Exercise 6 · One run is an anecdote

**Habit: many runs.** Ask Exercise 3's question ten times (each time is a fresh conversation) and count how often the answer names both *middle* and *south*. This cell is complete; it takes a few minutes. If your laptop is slow, change `RUNS` to 5 and say so in your answer.

In [ ]:
# Exercise 6: ten independent agent conversations, no fabricated responses.
# Prerequisite: run setup cells and Exercise 3 to define question_3 and run_agent.
import json
RUNS = 10
hits = 0
records = []
for i in range(RUNS):
    answer = run_agent(question_3, save=f"trace_ex6_{i}.json")
    text = answer or ""
    # Count only completed answers naming both crossings.
    good = text.strip().startswith("ANSWER:") and "middle" in text.lower() and "south" in text.lower()
    hits += int(good)
    records.append({"run": i + 1, "completed": text.strip().startswith("ANSWER:"),
                    "names_middle_and_south": good, "answer": text})
    print(f"run {i+1:2d}: {'PASS' if good else 'FAIL'} | {text[:140]}")
lo, hi = wilson(hits, RUNS)
print(f"\nExercise 6: {hits} of {RUNS} runs named middle + south; 95% Wilson interval {lo:.2f} to {hi:.2f}")
with open('exercise6_results.json', 'w') as f:
    json.dump({"runs": RUNS, "hits": hits, "wilson_95pct": [float(lo), float(hi)], "details": records}, f, indent=2)
print('Saved exercise6_results.json and trace_ex6_0.json through trace_ex6_9.json')


**Your answer (Ex 6):** After running the preceding cell, replace this sentence with the printed **hits / 10** and **95% Wilson interval**. Do not report a numerical result until the agent has actually run.

**Would I use the agent unsupervised?** No. It should be independently checked with consistent seeds, adequate simulation samples, and a review of its tool-call traces before informing any real decision.

**Two-sentence reflection:** Complete after reviewing your actual Exercise 3 trace: describe one specific action the agent took that surprised you. Before trusting a real-world recommendation, I would independently rerun every candidate plan under identical settings and assess uncertainty.

## What to hand in
- This notebook, executed top to bottom, with the observed agent comparisons filled in.
- `trace_ex3.json` (the Exercise 3 trace; check the exact filename saved in its cell).
- `exercise6_results.json` and ten Exercise 6 traces as supporting reproducibility evidence.
